## Step 1: Import and Inspect the Dataset

In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("website_analytics_dummy_dataset.csv")

# Quick overview
print(df.shape)
df.info()
df.head()


(3000, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Timestamp         3000 non-null   object 
 1   User_ID           3000 non-null   object 
 2   Page_Visited      3000 non-null   object 
 3   Device_Type       2850 non-null   object 
 4   Region            3000 non-null   object 
 5   Session_Duration  2851 non-null   float64
 6   Bounce_Rate       2850 non-null   float64
 7   Conversion        3000 non-null   int64  
 8   Page_Load_Time    2850 non-null   float64
dtypes: float64(3), int64(1), object(5)
memory usage: 211.1+ KB


,Timestamp,User_ID,Page_Visited,Device_Type,Region,Session_Duration,Bounce_Rate,Conversion,Page_Load_Time
0,1/1/2024 0:00,user_103,Home,Mobile,Europe,195.72,0.93,0,2.23
1,1/1/2024 0:15,user_436,Login,Mobile,North America,447.93,0.87,0,3.58
2,1/1/2024 0:30,user_861,Search,Tablet,Europe,582.88,0.13,0,0.97
3,1/1/2024 0:45,user_271,Login,Desktop,South America,265.63,0.17,0,9.53
4,1/1/2024 1:00,user_107,Checkout,Mobile,North America,517.31,0.36,0,6.02


This helps confirm:

Number of rows/columns

Data types

Presence of nulls or unexpected values

## Step 2: Identify Missing and Incorrect Data

In [ ]:
# Checking Missing Values/ Nulls
df.isnull().sum()

Timestamp             0
User_ID               0
Page_Visited          0
Device_Type         150
Region                0
Session_Duration    149
Bounce_Rate         150
Conversion            0
Page_Load_Time      150
dtype: int64

In [ ]:
# Percentage of Nulls

(df.isnull().sum()/len(df)) * 100

Timestamp           0.000000
User_ID             0.000000
Page_Visited        0.000000
Device_Type         5.000000
Region              0.000000
Session_Duration    4.966667
Bounce_Rate         5.000000
Conversion          0.000000
Page_Load_Time      5.000000
dtype: float64

In [4]:
df.isnull().mean().sort_values(ascending=False)

Device_Type         0.050000
Bounce_Rate         0.050000
Page_Load_Time      0.050000
Session_Duration    0.049667
Timestamp           0.000000
User_ID             0.000000
Page_Visited        0.000000
Region              0.000000
Conversion          0.000000
dtype: float64

In [5]:
# Empty strings

(df == "").sum()

Timestamp           0
User_ID             0
Page_Visited        0
Device_Type         0
Region              0
Session_Duration    0
Bounce_Rate         0
Conversion          0
Page_Load_Time      0
dtype: int64

No empty strings found

In [21]:
# whitespaces trailing/leading

col_values = [
    "Page_Visited", "Device_Type", "Region"
    ]

for col in col_values:
    series = df[col].astype(str)

    leading = series.str.startswith(" ").sum()
    trailing = series.str.endswith(" ").sum()
    print(" ")
    print(f"{col}:")
    print(f" Leading spaces: {leading} ")
    print(f" Trailing spaces: {trailing}")

print(" ")
# Detecting values that would change after stripping:
for col in col_values:
    series = df[col].astype(str)
    whitespace_issues = (series != series.str.strip()).sum()
    
    print(f"{col}: {whitespace_issues} values contain leading/trailing whitespace")

 
Page_Visited:
 Leading spaces: 0 
 Trailing spaces: 0
 
Device_Type:
 Leading spaces: 0 
 Trailing spaces: 0
 
Region:
 Leading spaces: 0 
 Trailing spaces: 0
 
Page_Visited: 0 values contain leading/trailing whitespace
Device_Type: 0 values contain leading/trailing whitespace
Region: 0 values contain leading/trailing whitespace


In [23]:
# Unique values check

col_values = [
    "Page_Visited", "Device_Type", "Region"
    ]

for col in col_values:
    series = df[col].astype(str)

    unique_val = series.value_counts(dropna = False)

# Case sensitive values

for col in col_values:
    series = df[col].astype(str)

    sensitive_val = series.str.lower().value_counts(dropna = False)


In [25]:
df.nunique()

Timestamp           3000
User_ID              934
Page_Visited           7
Device_Type            3
Region                 5
Session_Duration    2770
Bounce_Rate          102
Conversion             3
Page_Load_Time       900
dtype: int64

In [27]:
# mixed value check
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Timestamp         3000 non-null   object 
 1   User_ID           3000 non-null   object 
 2   Page_Visited      3000 non-null   object 
 3   Device_Type       2850 non-null   object 
 4   Region            3000 non-null   object 
 5   Session_Duration  2851 non-null   float64
 6   Bounce_Rate       2850 non-null   float64
 7   Conversion        3000 non-null   int64  
 8   Page_Load_Time    2850 non-null   float64
dtypes: float64(3), int64(1), object(5)
memory usage: 211.1+ KB


In [30]:
# detecting non-numeric values in numeric column
cols = [
    "Session_Duration", "Bounce_Rate", "Page_Load_Time" 
]

for col in cols:
    non_numeric_val = col[~col.astype(str).str.isnumeric()]

AttributeError: 'str' object has no attribute 'astype'

In [3]:
## Checking Invalid Entries

# Negative or impossible values
df[df["Session_Duration"] < 0]

# Bounce rate greater than 1
df[df["Bounce_Rate"] > 1]

# Invalid conversion flags
df[~df["Conversion"].isin([0, 1])]


,Timestamp,User_ID,Page_Visited,Device_Type,Region,Session_Duration,Bounce_Rate,Conversion,Page_Load_Time
160,2024-01-02 16:00:00,user_48,Checkout,Tablet,North America,207.92,0.30,2,6.66
246,2024-01-03 13:30:00,user_259,Login,Tablet,Asia,10.47,0.11,2,3.05
887,2024-01-10 05:45:00,user_220,Home,Desktop,Asia,NaN,0.34,2,1.23
1025,2024-01-11 16:15:00,user_385,Product,Desktop,Europe,257.88,0.82,2,5.47
1922,2024-01-21 00:30:00,user_616,Product,Tablet,Europe,25.10,0.26,2,3.28
2182,2024-01-23 17:30:00,user_195,Product,Desktop,Africa,NaN,0.67,2,8.80
2187,2024-01-23 18:45:00,user_521,Home,Mobile,Europe,407.11,0.05,2,4.51
2489,2024-01-26 22:15:00,user_672,Login,Desktop,Asia,137.45,0.63,2,3.23
2622,2024-01-28 07:30:00,user_516,Home,Desktop,Africa,499.69,0.98,2,NaN
2792,2024-01-30 02:00:00,user_428,Product,Desktop,Europe,368.06,0.18,2,5.32


## Step 3: Clean the Data

In [4]:
## a) Fix Missing Values

# Filling missing Device_Type with 'Unknown'
df["Device_Type"].fillna("Unknown", inplace=True)

# Filling missing numerical values with median
df["Session_Duration"].fillna(df["Session_Duration"].median(), inplace=True)
df["Bounce_Rate"].fillna(df["Bounce_Rate"].median(), inplace=True)
df["Page_Load_Time"].fillna(df["Page_Load_Time"].median(), inplace=True)


C:\Users\user\AppData\Local\Temp\ipykernel_15944\2982996679.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Device_Type"].fillna("Unknown", inplace=True)
C:\Users\user\AppData\Local\Temp\ipykernel_15944\2982996679.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

In [5]:
## b) Correct Invalid Values

# Replace negative session durations with median
df.loc[df["Session_Duration"] < 0, "Session_Duration"] = df["Session_Duration"].median()

# Clip bounce rate to valid range (0–1)
df["Bounce_Rate"] = df["Bounce_Rate"].clip(0, 1)

# Replace invalid conversions (2) with 0 (non-conversion)
df.loc[~df["Conversion"].isin([0, 1]), "Conversion"] = 0


In [6]:
## c) Validate Data Types

# Ensuring timestamp is datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Ensuring numeric columns are correct types
df = df.astype({
    "Session_Duration": "float64",
    "Bounce_Rate": "float64",
    "Conversion": "int64",
    "Page_Load_Time": "float64"
})


## Step 4: Feature Engineering for Dashboard Insights

In [7]:
# Extract time components
df["Date"] = df["Timestamp"].dt.date
df["Hour"] = df["Timestamp"].dt.hour
df["DayOfWeek"] = df["Timestamp"].dt.day_name()

# Engagement metrics
df["Engagement_Score"] = (df["Session_Duration"] / (df["Page_Load_Time"] + 1)).round(2)

# Conversion rate per session
df["Converted_Session"] = df["Conversion"].apply(lambda x: "Converted" if x == 1 else "Not Converted")


Now we can use these features for:

📊 Time-based metrics (e.g., daily/weekly activity)

💡 Performance metrics (load time vs bounce)

🧠 User behavior analysis (by region/device/time)

## Verifing Cleaned Data

In [9]:
df.isnull().sum()



Timestamp            0
User_ID              0
Page_Visited         0
Device_Type          0
Region               0
Session_Duration     0
Bounce_Rate          0
Conversion           0
Page_Load_Time       0
Date                 0
Hour                 0
DayOfWeek            0
Engagement_Score     0
Converted_Session    0
dtype: int64

In [10]:
df.describe()


,Timestamp,Session_Duration,Bounce_Rate,Conversion,Page_Load_Time,Hour,Engagement_Score
count,3000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,2024-01-16 14:52:29.999999744,297.282180,0.499110,0.097333,5.388113,11.428000,60.213993
min,2024-01-01 00:00:00,5.030000,0.000000,0.000000,0.510000,0.000000,0.580000
25%,2024-01-08 19:26:15,160.657500,0.260000,0.000000,3.210000,5.000000,25.167500
50%,2024-01-16 14:52:30,294.400000,0.490000,0.000000,5.460000,11.000000,45.445000
75%,2024-01-24 10:18:45,432.615000,0.750000,0.000000,7.572500,17.000000,76.387500
max,2024-02-01 05:45:00,599.710000,1.000000,1.000000,10.000000,23.000000,359.790000
std,NaN,164.744884,0.285033,0.296461,2.663860,6.943742,53.777811


In [11]:
df.head()

,Timestamp,User_ID,Page_Visited,Device_Type,Region,Session_Duration,Bounce_Rate,Conversion,Page_Load_Time,Date,Hour,DayOfWeek,Engagement_Score,Converted_Session
0,2024-01-01 00:00:00,user_103,Home,Mobile,Europe,195.72,0.93,0,2.23,2024-01-01,0,Monday,60.59,Not Converted
1,2024-01-01 00:15:00,user_436,Login,Mobile,North America,447.93,0.87,0,3.58,2024-01-01,0,Monday,97.80,Not Converted
2,2024-01-01 00:30:00,user_861,Search,Tablet,Europe,582.88,0.13,0,0.97,2024-01-01,0,Monday,295.88,Not Converted
3,2024-01-01 00:45:00,user_271,Login,Desktop,South America,265.63,0.17,0,9.53,2024-01-01,0,Monday,25.23,Not Converted
4,2024-01-01 01:00:00,user_107,Checkout,Mobile,North America,517.31,0.36,0,6.02,2024-01-01,1,Monday,73.69,Not Converted


## Step 6: Saving Clean Dataset for Dashboard Use

In [12]:
df.to_csv("cleaned_website_analytics_dataset.csv", index=False)
